# mBART Fine-tuning for English-to-Urdu Translation

Standalone notebook for fine-tuning Facebook's mBART-large-50 model on English-Urdu translation.

**Dataset:** Parallel Corpus for English-Urdu Language (24,525 sentence pairs)

## 1. Imports and Setup

In [1]:
import os
import re
import gc
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


## 2. Data Loading and Preprocessing

In [2]:
def load_data():
    en_path = "Dataset/english-corpus.txt"
    ur_path = "Dataset/urdu-corpus.txt"

    with open(en_path, "r", encoding="utf-8") as f:
        en_lines = [line.strip() for line in f.readlines()]
    with open(ur_path, "r", encoding="utf-8") as f:
        ur_lines = [line.strip() for line in f.readlines()]

    assert len(en_lines) == len(ur_lines), "Mismatch in number of lines"
    print(f"Loaded {len(en_lines)} sentence pairs")
    return en_lines, ur_lines


def clean_english(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'\-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_urdu(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preprocess_data(en_lines, ur_lines):
    pairs = []
    for en, ur in zip(en_lines, ur_lines):
        en_clean = clean_english(en)
        ur_clean = clean_urdu(ur)
        if len(en_clean) > 0 and len(ur_clean) > 0:
            pairs.append((en_clean, ur_clean))
    print(f"After cleaning: {len(pairs)} pairs")
    return pairs


en_lines, ur_lines = load_data()
pairs = preprocess_data(en_lines, ur_lines)

# Split data
random.seed(42)
random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.9)
n_val = int(n * 0.05)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

print(f"\nTrain: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")
print(f"\nSample pairs:")
for i in range(3):
    print(f"  EN: {train_pairs[i][0]}")
    print(f"  UR: {train_pairs[i][1]}")
    print()

Loaded 24525 sentence pairs
After cleaning: 24524 pairs

Train: 22071, Val: 1226, Test: 1227

Sample pairs:
  EN: run for it
  UR: اس کے لئے دوڑنا

  EN: are you blaming me
  UR: کیا تم مجھ پر الزام لگا رہے ہو

  EN: what did you guys do
  UR: كب سو گي آپ



## 3. Load mBART Model

In [3]:
# === AGGRESSIVE MEMORY OPTIMIZATION ===
model_name = "facebook/mbart-large-50"
mbart_tokenizer = MBart50TokenizerFast.from_pretrained(model_name)

# Load model
mbart_model = MBartForConditionalGeneration.from_pretrained(model_name)
mbart_model = mbart_model.cuda()

# Enable gradient checkpointing
mbart_model.gradient_checkpointing_enable()

mbart_tokenizer.src_lang = "en_XX"
mbart_tokenizer.tgt_lang = "ur_PK"

print("✓ Model loaded with gradient checkpointing")

d:\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.00 MB. The target location C:\Users\Lenovo\.cache\huggingface\hub\models--facebook--mbart-large-50\blobs only has 0.00 MB free disk space.
  warnings.warn(
d:\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--facebook--mbart-large-50. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an a

OSError: Can't load tokenizer for 'facebook/mbart-large-50'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'facebook/mbart-large-50' is the correct path to a directory containing all relevant files for a MBart50Tokenizer tokenizer.

## 4. Dataset Class

In [ ]:
class MBartTranslationDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=32):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, ur = self.pairs[idx]
        
        model_inputs = self.tokenizer(
            en, 
            max_length=self.max_len, 
            truncation=True, 
            padding="max_length"
        )
        
        labels = self.tokenizer(
            text_target=ur, 
            max_length=self.max_len, 
            truncation=True, 
            padding="max_length"
        )

        return {
            "input_ids": torch.tensor(model_inputs["input_ids"]).squeeze(),
            "attention_mask": torch.tensor(model_inputs["attention_mask"]).squeeze(),
            "labels": torch.tensor(labels["input_ids"]).squeeze(),
        }


# Use subset of data for faster training
print(f"Full train set: {len(train_pairs)} samples")
mbart_train = MBartTranslationDataset(train_pairs[:5000], mbart_tokenizer)
mbart_val = MBartTranslationDataset(val_pairs[:1000], mbart_tokenizer)

print(f"Train samples used: {len(mbart_train)}")
print(f"Val samples used: {len(mbart_val)}")

## 5. Training Configuration and Setup

In [ ]:
# === ULTRA-MEMORY-EFFICIENT SETTINGS ===
training_args = Seq2SeqTrainingArguments(
    output_dir="mbart_checkpoints",
    num_train_epochs=4,              
    
    per_device_train_batch_size=2,   # Batch size 
    gradient_accumulation_steps=1,   # NO accumulation
    
    per_device_eval_batch_size=1,
    eval_strategy="no",              # NO evaluation
    save_strategy="no",              # NO checkpoints during training
    
    learning_rate=2e-5,
    weight_decay=0.01,
    predict_with_generate=False,
    
    logging_steps=100,
    report_to="none",
    seed=42,
    remove_unused_columns=False,
    max_steps=1000,                  # Limit to 1000 steps
)

data_collator = DataCollatorForSeq2Seq(mbart_tokenizer, model=mbart_model)

trainer = Seq2SeqTrainer(
    model=mbart_model,
    args=training_args,
    train_dataset=mbart_train,
    eval_dataset=None,
    data_collator=data_collator,
)

print("✓ Trainer configured")
print("  - Batch size: 1")
print("  - Max length: 32")
print("  - Limited to 1000 steps")
print("  - No evaluation during training")

## 6. Train the Model

In [ ]:
print("\n" + "="*70)
print("Starting mBART Fine-tuning...")
print("="*70 + "\n")

trainer.train()

trainer.save_model("mbart_finetuned")
print("\n✓ Training complete! Model saved to 'mbart_finetuned/")

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()
print("✓ GPU memory cleared")

## 7. Inference Function

In [ ]:
def translate_mbart(text, model, tokenizer):
    """Translate English text to Urdu using fine-tuned mBART."""
    tokenizer.src_lang = "en_XX"
    inputs = tokenizer(text, return_tensors="pt", max_length=32, truncation=True).to(model.device)
    generated = model.generate(
        **inputs, 
        forced_bos_token_id=tokenizer.lang_code_to_id["ur_PK"], 
        max_length=32,
        num_beams=1  # Greedy decoding
    )
    return tokenizer.decode(generated[0], skip_special_tokens=True)

print("✓ Inference function defined")

## 8. Test Translations

In [ ]:
test_sentences = [
    "how are you",
    "what is your name",
    "i love my country",
    "the weather is beautiful today",
    "where is the school",
    "he is a good person",
    "please help me",
    "i am going home",
]

print("\n" + "="*70)
print("mBART English-to-Urdu Translations (Fine-tuned)")
print("="*70)
for sent in test_sentences:
    translation = translate_mbart(sent, mbart_model, mbart_tokenizer)
    print(f"EN: {sent}")
    print(f"UR: {translation}")
    print("-" * 70)

## 9. Evaluate on Test Set

In [ ]:
def compute_bleu(reference, hypothesis, max_n=4):
    """Compute BLEU score for a single reference-hypothesis pair."""
    from collections import Counter
    import math
    
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if len(hyp_tokens) == 0:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter()
        for i in range(len(ref_tokens) - n + 1):
            ngram = tuple(ref_tokens[i:i+n])
            ref_ngrams[ngram] += 1

        hyp_ngrams = Counter()
        for i in range(len(hyp_tokens) - n + 1):
            ngram = tuple(hyp_tokens[i:i+n])
            hyp_ngrams[ngram] += 1

        clipped = sum(min(hyp_ngrams[ng], ref_ngrams[ng]) for ng in hyp_ngrams)
        total = max(sum(hyp_ngrams.values()), 1)
        precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    bp = 1.0
    if len(hyp_tokens) < len(ref_tokens):
        bp = math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))

    return bp * math.exp(log_avg)


def corpus_bleu(references, hypotheses):
    """Compute average BLEU score over a corpus."""
    scores = [compute_bleu(ref, hyp) for ref, hyp in zip(references, hypotheses)]
    return sum(scores) / max(len(scores), 1)


# Evaluate on first 100 test samples
print("\nEvaluating on test set...")
references, hypotheses = [], []
for en_s, ur_s in test_pairs[:100]:
    pred = translate_mbart(en_s, mbart_model, mbart_tokenizer)
    references.append(ur_s)
    hypotheses.append(pred)

test_bleu = corpus_bleu(references, hypotheses)
print(f"\nTest BLEU Score (first 100 samples): {test_bleu:.4f}")
print(f"Total test samples: {len(test_pairs)}")

print("\n" + "="*80)
print(f"{'English':<35} | {'Predicted Urdu':<35}")
print("="*80)
for i in range(min(10, len(test_pairs))):
    en_s = test_pairs[i][0]
    ref = test_pairs[i][1]
    hyp = hypotheses[i]
    print(f"EN:   {en_s}")
    print(f"REF:  {ref}")
    print(f"PRED: {hyp}")
    print("-"*80)